# Data Preparation — DDD-Cambodia Dataset

**Notebook purpose:** Clean, verify, and format the DDD-Cambodia Khmer ASR Cultural
Dataset before it's merged into the final training set. Confirms sample rate,
audio-transcript alignment, and spot-checks pairing correctness.

## Related task
+ `[DATA-05] Download and verify the primary additional training dataset.` (see Trello board)
+ `[DATA-07] Clean & Format DDD-Cambodia Data` (see Trello board)
+ `[DATA-08] Generate synthetic "phone-mic quality" training data from clean audio.` (see Trello board)
`[DATA-09] Merge & Finalize Training Dataset` (see Trello board)

In [ ]:
# Library
import os
from datasets import load_dataset
from huggingface_hub import DatasetCard

### Mount to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Create folder in drive

In [ ]:
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/khmer-stt-bot'
os.makedirs(f'{DRIVE_PROJECT_DIR}/data/raw', exist_ok=True)
print("Drive mounted. Project folder ready at:", DRIVE_PROJECT_DIR)

### Load the dataset (from Hugging Face) and Save raw files to data/raw/

In [ ]:
!df -h /

In [ ]:
ddd_stream = load_dataset("DDD-Cambodia/khm-asr-cultural", split="train", streaming=True)
print("Streaming dataset ready — no full download yet, just a connection to it")

In [ ]:
import os, soundfile as sf, csv

DDD_RAW_DIR = f"{DRIVE_PROJECT_DIR}/data/raw/ddd_cambodia"
os.makedirs(DDD_RAW_DIR, exist_ok=True)

In [ ]:
metadata_path = f"{DDD_RAW_DIR}/metadata.tsv"
existing_files = set()

# Resume support — skip files already saved if this is a re-run
if os.path.exists(metadata_path):
    with open(metadata_path, encoding="utf-8") as f:
        existing_files = {line.split("\t")[0] for line in f}
    print(f"Resuming — {len(existing_files)} files already saved")

with open(metadata_path, "a", encoding="utf-8") as meta_file:
    for i, sample in enumerate(ddd_stream):
        filename = f"ddd_{i:05d}.wav"
        if filename in existing_files:
            continue

        audio = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]
        text = sample["transcription"] if "transcription" in sample else sample.get("raw_transcription", "")

        filepath = os.path.join(DDD_RAW_DIR, filename)
        sf.write(filepath, audio, sr)
        meta_file.write(f"{filename}\t{text}\n")
        meta_file.flush()  # write immediately, don't buffer

        if i % 200 == 0:
            print(f"  progress: {i} samples saved")

print("Done streaming and saving to Drive")

### Verify the license terms

In [ ]:
# Check the dataset card / README for license confirmation
card = DatasetCard.load("DDD-Cambodia/khm-asr-cultural")
print(card.data.license)

## Step: Clean & Format DDD-Cambodia Data

Verify the downloaded DDD-Cambodia dataset is ready for training — confirm audio
format, check audio-transcript alignment, and spot-check pairing correctness before
this data is used in fine-tuning.

**Findings:**
- Sample rate check (50-file sample) confirmed all audio is already 16kHz mono —
  no resampling needed.
- Since no transformation was required, `data/raw/ddd_cambodia/` is used directly
  as the processed dataset. No separate copy was created in `data/processed/` to
  avoid duplicating ~30GB of identical data for no benefit. (See commented-out
  copy code below, kept for structural reference.)
- Alignment check confirms metadata pair count matches audio file count
  (56,716 both).

### Check what I actually have first

In [ ]:
import soundfile as sf
import glob

wav_files = glob.glob(f"{DDD_RAW_DIR}/*.wav")
print(f"Found {len(wav_files)} files")

In [ ]:
total_seconds = 0
for f in wav_files:
    info = sf.info(f)
    total_seconds += info.duration

total_hours = total_seconds / 3600
print(f"Total duration: {total_hours:.2f} hours")

In [ ]:
# Check a sample of sample rates — not all 56,600, just a quick spot-check
sample_rates = set()
for f in wav_files[:50]:
    info = sf.info(f)
    sample_rates.add(info.samplerate)

print("Sample rates found in first 50 files:", sample_rates)

### Resample only if needed

In [ ]:
# import os
# import librosa

# PROCESSED_DIR = f"{DRIVE_PROJECT_DIR}/data/processed/ddd_cambodia"
# os.makedirs(PROCESSED_DIR, exist_ok=True)

# for i, f in enumerate(wav_files):
#     audio, sr = librosa.load(f, sr=16000, mono=True)  # forces 16kHz mono on load
#     filename = os.path.basename(f)
#     sf.write(os.path.join(PROCESSED_DIR, filename), audio, 16000)

#     if i % 2000 == 0:
#         print(f"  resampled {i}/{len(wav_files)}")

# print("Done resampling into data/processed/ddd_cambodia")

### Copy to data/processed/ (no re-encoding needed)

In [ ]:
# import os
# import shutil

# PROCESSED_DIR = f"{DRIVE_PROJECT_DIR}/data/processed/ddd_cambodia"
# os.makedirs(PROCESSED_DIR, exist_ok=True)

# source_files = os.listdir(DDD_RAW_DIR)
# copied = 0
# skipped = 0

# for i, filename in enumerate(source_files):
#     src = os.path.join(DDD_RAW_DIR, filename)
#     dst = os.path.join(PROCESSED_DIR, filename)

#     # Skip if already copied AND same size — genuine resume, not blind overwrite
#     if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
#         skipped += 1
#         continue

#     shutil.copy2(src, dst)
#     copied += 1

#     if i % 2000 == 0:
#         print(f"  progress: {i}/{len(source_files)} (copied: {copied}, skipped: {skipped})")

# print(f"Done. Copied: {copied}, Skipped (already done): {skipped}")

### Spot-check a handful for real

In [ ]:
# import random

# sample_indices = random.sample(range(len(pairs)), 5)

# for idx in sample_indices:
#     filename, text = pairs[idx]
#     filepath = os.path.join(PROCESSED_DIR, filename)
#     duration = sf.info(filepath).duration
#     print(f"File: {filename}")
#     print(f"  Transcript: {text}")
#     print(f"  Duration: {duration:.2f}s")
#     print()

**Note:** DDD-Cambodia audio was already 16kHz mono on download — no resampling required.
data/raw/ddd_cambodia/ serves directly as the processed dataset; no separate copy
was created in data/processed/ to avoid duplicating ~30GB for no transformation.

### Confirm audio↔transcript alignment

In [ ]:
import csv

metadata_path = f"{DDD_RAW_DIR}/metadata.tsv"

pairs = []
problem_lines = []

with open(metadata_path, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
    for i, row in enumerate(reader):
        if len(row) < 2:
            problem_lines.append((i, row))
            continue
        pairs.append((row[0], row[1]))

print(f"Total pairs in metadata: {len(pairs)}")
print(f"Total raw audio files: {len(glob.glob(f'{DDD_RAW_DIR}/*.wav'))}")
print(f"Problem lines skipped: {len(problem_lines)}")

if problem_lines:
    print("First few problem lines:", problem_lines[:5])

## Step: Build Audio Augmentation Pipeline

Generate synthetic "phone-mic quality" versions of a subset of the clean DDD-Cambodia
audio, to teach the fine-tuned model robustness to real-world recording conditions
(background noise, gain variation, compression artifacts) — since real Telegram voice
notes won't sound like clean studio-quality recordings.

**Approach:**
1. Build and test the augmentation chain (noise + gain + simulated low-bitrate
   compression) on a single sample first, listen to confirm it sounds like a
   realistic phone-mic recording (not pure noise, not imperceptibly different).
2. Once confirmed, apply to a subset of the dataset (not all 56,716 files, to avoid
   unnecessarily doubling storage/training time) — augmented copies stored
   alongside the clean originals in `data/processed/`.


In [ ]:
!pip install -q audiomentations

### Write the augmentation function — test on ONE file first

In [ ]:
from audiomentations import Compose, AddGaussianNoise, ApplyImpulseResponse, Gain, PitchShift
import numpy as np
import soundfile as sf

# Simulates: background noise + gain variation (loud/quiet mic pickup) + slight pitch
# instability — approximates a cheap phone mic recording in a normal environment

augment = Compose([
    AddGaussianNoise(
        min_amplitude=0.003,
        max_amplitude=0.015,
        p=0.8
    ),

    Gain(
        min_gain_db=-6,
        max_gain_db=6,
        p=0.7
    ),

    PitchShift(
        min_semitones=-1,
        max_semitones=1,
        p=0.2
    ),
])


# Load one sample to test on
test_file = wav_files[0]  # reuse the list from earlier
audio, sr = sf.read(test_file)

augmented_audio = augment(samples=audio.astype(np.float32), sample_rate=sr)

sf.write("test_augmented.wav", augmented_audio, sr)
print("Saved test_augmented.wav — listen to it below")

### Listen to both, side by side

In [ ]:
from IPython.display import Audio, display

print("Original:")
display(Audio(test_file))

print("Augmented (simulated phone-mic):")
display(Audio("test_augmented.wav"))

### **Simulating bitrate reduction** (the "compressed phone audio" effect) needs one more step - actually encoding through a lossy codec and back, since audiomentations doesn't fake this directly

In [ ]:
import subprocess

def simulate_low_bitrate(input_path, output_path, bitrate="16k"):
    # Encode to a low-bitrate .ogg then back to .wav — mimics compression artifacts
    # from Telegram-style voice messages
    temp_ogg = input_path.replace(".wav", "_temp.ogg")
    subprocess.run(["ffmpeg", "-y", "-i", input_path, "-b:a", bitrate, temp_ogg],
                   capture_output=True)
    subprocess.run(["ffmpeg", "-y", "-i", temp_ogg, output_path],
                   capture_output=True)
    os.remove(temp_ogg)

simulate_low_bitrate("test_augmented.wav", "test_augmented_lowbitrate.wav")
display(Audio("test_augmented_lowbitrate.wav"))

### Once it sounds right — decide on scope before generating thousands of files

In [ ]:
import random

# Augment ~20% of the dataset — enough to teach the model phone-mic robustness
# without doubling total storage/training time unnecessarily
AUGMENT_FRACTION = 0.2
files_to_augment = random.sample(wav_files, int(len(wav_files) * AUGMENT_FRACTION))
print(f"Will augment {len(files_to_augment)} of {len(wav_files)} files")

### The full batch augmentation loop

In [ ]:
import os
import csv

AUGMENTED_DIR = f"{DRIVE_PROJECT_DIR}/data/processed/ddd_augmented"
os.makedirs(AUGMENTED_DIR, exist_ok=True)

# Load original transcripts so augmented files can reference the same text
# (augmentation changes the audio, not what was said)
original_metadata = {}
with open(f"{DDD_RAW_DIR}/metadata.tsv", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        original_metadata[row[0]] = row[1]

aug_metadata_path = f"{AUGMENTED_DIR}/metadata.tsv"
existing_augmented = set()

# Resume support
if os.path.exists(aug_metadata_path):
    with open(aug_metadata_path, encoding="utf-8") as f:
        existing_augmented = {line.split("\t")[0] for line in f}
    print(f"Resuming — {len(existing_augmented)} files already augmented")

In [ ]:
with open(aug_metadata_path, "a", encoding="utf-8") as meta_file:
    for i, src_path in enumerate(files_to_augment):
        original_filename = os.path.basename(src_path)
        aug_filename = f"aug_{original_filename}"

        if aug_filename in existing_augmented:
            continue

        # Step 1: apply noise/gain/pitch augmentation
        audio, sr = sf.read(src_path)
        augmented_audio = augment(samples=audio.astype(np.float32), sample_rate=sr)

        temp_wav = f"/content/{aug_filename}"
        sf.write(temp_wav, augmented_audio, sr)

        # Step 2: simulate low-bitrate compression on top
        final_path = os.path.join(AUGMENTED_DIR, aug_filename)
        simulate_low_bitrate(temp_wav, final_path)
        os.remove(temp_wav)  # clean up the intermediate local file

        # Reuse the original transcript — same words, different audio quality
        transcript = original_metadata[original_filename]
        meta_file.write(f"{aug_filename}\t{transcript}\n")
        meta_file.flush()

        if i % 500 == 0:
            print(f"  progress: {i}/{len(files_to_augment)}")

print(f"Done. Augmented files saved to {AUGMENTED_DIR}")

### sanity check

In [ ]:
from IPython.display import Audio, display
import random

aug_files = os.listdir(AUGMENTED_DIR)
sample_check = random.sample([f for f in aug_files if f.endswith('.wav')], 3)

for f in sample_check:
    print(f)
    display(Audio(os.path.join(AUGMENTED_DIR, f)))

## Step: Merge & Finalize Training Dataset

Combine all training data sources into a single, traceable dataset ready for
fine-tuning: the data already inherited in the `whisper-small-khmer-v2` checkpoint
(FLEURS + OpenSLR SLR42 + km-speech-corpus) plus the newly acquired DDD-Cambodia
Khmer ASR Cultural Dataset and its augmented ("phone-mic quality") counterpart.

**What this step does:**
1. Builds a unified manifest listing every training file, its transcript, and which
   source it came from — keeps composition traceable for the thesis write-up.
2. Deduplicates and shuffles the combined set.
3. Documents the final composition (source breakdown, total hours) in `data/DATASETS.md`.

**Note:** the inherited sources (FLEURS, OpenSLR, km-speech-corpus) are not
re-downloaded here — they're already baked into the `v2` checkpoint we're fine-tuning
from. This step only combines the *new* data (DDD-Cambodia raw + augmented) that gets
added on top during fine-tuning.

### **Build Unified Training Manifest (Original + Augmented Data)**

In [ ]:
import csv

manifest_path = f"{DRIVE_PROJECT_DIR}/data/processed/training_manifest.tsv"

sources = [
    (DDD_RAW_DIR, "ddd_raw"),
    (AUGMENTED_DIR, "ddd_augmented"),
]

skipped_empty = 0
written = 0

with open(manifest_path, "w", encoding="utf-8") as out:
    for folder, source_label in sources:
        metadata_file = f"{folder}/metadata.tsv"
        with open(metadata_file, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")  # only strip the newline, not tabs
                parts = line.split("\t", 1)

                if len(parts) < 2 or not parts[1].strip():
                    skipped_empty += 1
                    continue  # skip entries with no real transcript

                filename, text = parts
                out.write(f"{filename}\t{text}\t{source_label}\n")
                written += 1

print(f"Manifest built: {manifest_path}")
print(f"Written: {written} entries")
print(f"Skipped (empty/malformed transcript): {skipped_empty} entries")

In [ ]:
for folder, source_label in sources:
    metadata_file = f"{folder}/metadata.tsv"
    with open(metadata_file, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(source_label, repr(line))

In [ ]:
from datasets import load_dataset

# Load just enough to inspect one sample's actual structure — don't need
# the whole streaming download again, just a peek
ddd_check = load_dataset("DDD-Cambodia/khm-asr-cultural", split="train", streaming=True)
sample = next(iter(ddd_check))

print("Available fields:", list(sample.keys()))
print()
for key, value in sample.items():
    if key != "audio":  # skip printing the raw audio array
        print(f"{key}: {value}")

#### **The fix**: rebuild just the metadata, reusing the audio you already have

In [ ]:
correct_metadata_path = f"{DDD_RAW_DIR}/metadata.tsv"

In [ ]:
from datasets import load_dataset

# Stream again, but drop the audio column entirely — this makes it fast,
# since we're not decoding/downloading audio, just reading text fields
ddd_text_stream = load_dataset("DDD-Cambodia/khm-asr-cultural", split="train", streaming=True)
ddd_text_stream = ddd_text_stream.remove_columns(["audio"])

with open(correct_metadata_path, "w", encoding="utf-8") as meta_file:
    for i, sample in enumerate(ddd_text_stream):
        filename = f"ddd_{i:05d}.wav"
        transcript = sample["transcript"]
        meta_file.write(f"{filename}\t{transcript}\n")

        if i % 5000 == 0:
            print(f"  rebuilt {i} entries")

print("Rebuilt metadata.tsv with correct transcripts")

### Verify the fix actually worked before trusting it

In [ ]:
with open(correct_metadata_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(repr(line))

### Then rebuild the augmented metadata too (reusing the corrected raw transcripts)

In [ ]:
AUGMENTED_DIR = f"{DRIVE_PROJECT_DIR}/data/processed/ddd_augmented"
os.makedirs(AUGMENTED_DIR, exist_ok=True)

In [ ]:
import os
import csv

original_metadata = {}
skipped = 0

with open(correct_metadata_path, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
    for row in reader:
        if len(row) < 2 or not row[1].strip():
            skipped += 1
            continue
        original_metadata[row[0]] = row[1]

print(f"Loaded {len(original_metadata)} valid raw transcripts (skipped {skipped})")

aug_metadata_path = f"{AUGMENTED_DIR}/metadata.tsv"
aug_files = [f for f in os.listdir(AUGMENTED_DIR) if f.endswith(".wav")]

matched = 0
with open(aug_metadata_path, "w", encoding="utf-8") as meta_file:
    for aug_filename in aug_files:
        original_filename = aug_filename.replace("aug_", "", 1)
        if original_filename in original_metadata:
            meta_file.write(f"{aug_filename}\t{original_metadata[original_filename]}\n")
            matched += 1

print(f"Rebuilt augmented metadata for {matched} of {len(aug_files)} files")

### One important sanity check before fully trusting the index-matching

In [ ]:
import soundfile as sf

# Spot-check: does the re-streamed order actually match the saved audio files?
ddd_verify = load_dataset("DDD-Cambodia/khm-asr-cultural", split="train", streaming=True)

for i, sample in enumerate(ddd_verify):
    if i >= 5:
        break
    expected_duration = sample["duration"]
    actual_path = f"{DDD_RAW_DIR}/ddd_{i:05d}.wav"
    actual_duration = sf.info(actual_path).duration
    match = abs(expected_duration - actual_duration) < 0.5
    print(f"Index {i}: expected {expected_duration:.2f}s, actual {actual_duration:.2f}s — {'✓ match' if match else '✗ MISMATCH'}")

### **Rebuild Unified Training Manifest (Original + Augmented Data)**

In [ ]:
import csv

manifest_path = f"{DRIVE_PROJECT_DIR}/data/processed/training_manifest.tsv"

sources = [
    (DDD_RAW_DIR, "ddd_raw"),
    (AUGMENTED_DIR, "ddd_augmented"),
]

skipped_empty = 0
written = 0

with open(manifest_path, "w", encoding="utf-8") as out:
    for folder, source_label in sources:
        metadata_file = f"{folder}/metadata.tsv"
        with open(metadata_file, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")  # only strip the newline, not tabs
                parts = line.split("\t", 1)

                if len(parts) < 2 or not parts[1].strip():
                    skipped_empty += 1
                    continue  # skip entries with no real transcript

                filename, text = parts
                out.write(f"{filename}\t{text}\t{source_label}\n")
                written += 1

print(f"Manifest built: {manifest_path}")
print(f"Written: {written} entries")
print(f"Skipped (empty/malformed transcript): {skipped_empty} entries")

### **Deduplicate and shuffle**

In [ ]:
import random

# Read the manifest back in
entries = []
with open(manifest_path, encoding="utf-8") as f:
    for line in f:
        entries.append(line.strip())

print(f"Before dedup: {len(entries)} entries")

# Deduplicate by filename (first column) — shouldn't have real duplicates given
# how files were named, but this catches any accidental double-processing
seen_filenames = set()
deduped = []
for entry in entries:
    filename = entry.split("\t")[0]
    if filename not in seen_filenames:
        seen_filenames.add(filename)
        deduped.append(entry)

print(f"After dedup: {len(deduped)} entries")

# Shuffle so training doesn't see all-raw-then-all-augmented in sequence
random.seed(42)  # fixed seed — reproducible shuffle, worth noting in thesis
random.shuffle(deduped)

with open(manifest_path, "w", encoding="utf-8") as f:
    f.write("\n".join(deduped))

print("Manifest deduplicated and shuffled.")

### **Update data/DATASETS.md with final composition**

In [ ]:
# Quick composition summary to help fill in DATASETS.md
from collections import Counter

source_counts = Counter()
with open(manifest_path, encoding="utf-8") as f:
    for line in f:
        source = line.strip().split("\t")[-1]
        source_counts[source] += 1

for source, count in source_counts.items():
    print(f"{source}: {count} files")
print(f"Total: {sum(source_counts.values())} files")